# Clusterização de Cidades (KMeans, DBSCAN, Aglomerativo, Espectral)

Este notebook:
- Padroniza os dados
- Testa KMeans com diferentes K e calcula **silhouette_score**
- Testa **DBSCAN** (com grid simples de parâmetros) e calcula **silhouette_score** (sem ruído)
- Testa **Aglomerativo** e **Espectral**
- Compara os métodos e mostra o melhor score
- Visualiza clusters em 2D com PCA


In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
sns.set(style='whitegrid')


## 1) Inputs (mude aqui para testar)

In [ ]:
# ========= INPUTS PRINCIPAIS =========
DATA_PATH = '/content/city_lifestyle_dataset.csv'

# K para métodos que precisam (KMeans / Aglomerativo / Espectral)
K = 4

# Intervalo para buscar melhor K no KMeans
K_RANGE = range(2, 11)  # 2..10

# Grid simples de DBSCAN (você pode expandir)
DBSCAN_EPS_LIST = [0.6, 0.8, 1.0, 1.2, 1.5]
DBSCAN_MIN_SAMPLES_LIST = [3, 5, 8, 10]

# PCA para visualização
PCA_COMPONENTS = 2

# =====================================
print('Inputs carregados:')
print(f'- DATA_PATH={DATA_PATH}')
print(f'- K={K}')
print(f'- K_RANGE={list(K_RANGE)}')
print(f'- DBSCAN_EPS_LIST={DBSCAN_EPS_LIST}')
print(f'- DBSCAN_MIN_SAMPLES_LIST={DBSCAN_MIN_SAMPLES_LIST}')


## 2) Leitura e preparação dos dados

In [ ]:
df = pd.read_csv(DATA_PATH)

display(df.head())
print('\nShape:', df.shape)
print('\nColunas:', df.columns.tolist())

# Features numéricas
X = df.drop(['city_name', 'country'], axis=1)

# Padronização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('\nX shape:', X.shape)
print('X_scaled shape:', X_scaled.shape)


## 3) Funções utilitárias (silhouette, PCA plot, resumo de clusters)

In [ ]:
def safe_silhouette(X_scaled, labels):
    """Calcula silhouette_score quando aplicável.
    Retorna np.nan se não for aplicável (ex.: 1 cluster).
    """
    labels = np.asarray(labels)
    unique = set(labels)
    # precisa de >=2 clusters
    if len(unique) < 2:
        return np.nan
    return silhouette_score(X_scaled, labels)


def safe_silhouette_dbscan(X_scaled, labels):
    """Silhouette para DBSCAN removendo ruído (-1).
    Retorna np.nan se não houver >=2 clusters sem ruído.
    """
    labels = np.asarray(labels)
    mask = labels != -1
    if mask.sum() <= 1:
        return np.nan
    unique = set(labels[mask])
    if len(unique) < 2:
        return np.nan
    return silhouette_score(X_scaled[mask], labels[mask])


def plot_pca_clusters(X_scaled, labels, title, df_ref=None):
    pca = PCA(n_components=PCA_COMPONENTS)
    X_pca = pca.fit_transform(X_scaled)

    pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(PCA_COMPONENTS)])
    pca_df['cluster'] = labels
    if df_ref is not None:
        pca_df['city_name'] = df_ref['city_name'].values
        pca_df['country'] = df_ref['country'].values

    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        data=pca_df,
        x='PC1', y='PC2',
        hue='cluster',
        palette='viridis',
        s=80,
        alpha=0.8
    )
    plt.title(title)
    plt.grid(True)
    plt.show()
    return pca_df


def print_cluster_summary(df_ref, labels, top_n=5):
    tmp = df_ref.copy()
    tmp['cluster'] = labels

    print('Top cidades por cluster:')
    for cid in sorted(tmp['cluster'].unique()):
        cities = tmp[tmp['cluster'] == cid]['city_name'].head(top_n).tolist()
        print(f"  Cluster {cid}: {', '.join(cities)}")

    print('\nPaíses por cluster:')
    for cid in sorted(tmp['cluster'].unique()):
        countries = tmp[tmp['cluster'] == cid]['country'].unique().tolist()
        print(f"  Cluster {cid}: {', '.join(countries)}")


## 4) KMeans com K como input + silhouette_score

In [ ]:
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_scaled)
sil_kmeans = safe_silhouette(X_scaled, labels_kmeans)

print(f"KMeans | K={K} | silhouette_score={sil_kmeans:.4f}")
print_cluster_summary(df, labels_kmeans, top_n=5)

pca_df_kmeans = plot_pca_clusters(X_scaled, labels_kmeans, title=f'KMeans (K={K}) - PCA 2D', df_ref=df)


## 5) KMeans: testar vários K e encontrar o melhor silhouette

In [ ]:
kmeans_results = []
for k in K_RANGE:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = safe_silhouette(X_scaled, labels)
    kmeans_results.append({'method': 'KMeans', 'k': k, 'silhouette': score})

kmeans_results_df = pd.DataFrame(kmeans_results).sort_values('silhouette', ascending=False)
display(kmeans_results_df)

best_kmeans = kmeans_results_df.iloc[0]
print(f"Melhor KMeans no intervalo {list(K_RANGE)}: K={int(best_kmeans['k'])} com silhouette={best_kmeans['silhouette']:.4f}")


## 6) DBSCAN: grid de parâmetros + melhor silhouette (sem ruído)

In [ ]:
dbscan_results = []

for eps in DBSCAN_EPS_LIST:
    for ms in DBSCAN_MIN_SAMPLES_LIST:
        model = DBSCAN(eps=eps, min_samples=ms)
        labels = model.fit_predict(X_scaled)

        n_noise = int((labels == -1).sum())
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        score = safe_silhouette_dbscan(X_scaled, labels)
        dbscan_results.append({
            'method': 'DBSCAN',
            'eps': eps,
            'min_samples': ms,
            'clusters': n_clusters,
            'noise_points': n_noise,
            'silhouette': score,
        })

dbscan_results_df = pd.DataFrame(dbscan_results)
dbscan_results_df_sorted = dbscan_results_df.sort_values('silhouette', ascending=False)
display(dbscan_results_df_sorted)

best_dbscan = dbscan_results_df_sorted.iloc[0]
print('Melhor DBSCAN (neste grid):')
print(best_dbscan)

# Rodar novamente DBSCAN com o melhor (para plot)
best_db = DBSCAN(eps=float(best_dbscan['eps']), min_samples=int(best_dbscan['min_samples']))
labels_dbscan = best_db.fit_predict(X_scaled)

sil_dbscan = safe_silhouette_dbscan(X_scaled, labels_dbscan)
print(f"DBSCAN | eps={best_dbscan['eps']} | min_samples={best_dbscan['min_samples']} | silhouette(no-noise)={sil_dbscan:.4f}")

pca_df_dbscan = plot_pca_clusters(X_scaled, labels_dbscan, title='DBSCAN (melhor do grid) - PCA 2D', df_ref=df)


## 7) Aglomerativo (Hierárquico) com K como input

In [ ]:
agg = AgglomerativeClustering(n_clusters=K, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)
sil_agg = safe_silhouette(X_scaled, labels_agg)

print(f"Aglomerativo | K={K} | silhouette_score={sil_agg:.4f}")
print_cluster_summary(df, labels_agg, top_n=5)

pca_df_agg = plot_pca_clusters(X_scaled, labels_agg, title=f'Aglomerativo (K={K}) - PCA 2D', df_ref=df)


## 8) Espectral com K como input

In [ ]:
spec = SpectralClustering(
    n_clusters=K,
    random_state=42,
    affinity='nearest_neighbors',
    n_neighbors=10
)
labels_spec = spec.fit_predict(X_scaled)
sil_spec = safe_silhouette(X_scaled, labels_spec)

print(f"Espectral | K={K} | silhouette_score={sil_spec:.4f}")
print_cluster_summary(df, labels_spec, top_n=5)

pca_df_spec = plot_pca_clusters(X_scaled, labels_spec, title=f'Espectral (K={K}) - PCA 2D', df_ref=df)


## 9) Comparação final: qual o melhor silhouette?

In [ ]:
comparison = []
comparison.append({'method': 'KMeans', 'params': f'K={K}', 'silhouette': sil_kmeans})
comparison.append({'method': 'Aglomerativo', 'params': f'K={K}', 'silhouette': sil_agg})
comparison.append({'method': 'Espectral', 'params': f'K={K}', 'silhouette': sil_spec})

# DBSCAN melhor do grid
comparison.append({
    'method': 'DBSCAN',
    'params': f"eps={best_dbscan['eps']}, min_samples={best_dbscan['min_samples']}, clusters={int(best_dbscan['clusters'])}, noise={int(best_dbscan['noise_points'])}",
    'silhouette': sil_dbscan
})

comparison_df = pd.DataFrame(comparison).sort_values('silhouette', ascending=False)
display(comparison_df)

best = comparison_df.iloc[0]
print(f"\nMelhor método (pelo silhouette): {best['method']} | {best['params']} | silhouette={best['silhouette']:.4f}")
